# Onion Quality Assessment — End-to-End Colab Training & Local Export
**Handheld Onion Quality Assessment & Multi-Standard Grading (DoCA + AGMARK)**

This workbook runs in **Google Colab via the VS Code extension**.
It executes the complete Phase 2 model training pipeline:
- **Model 1**: YOLOv8-seg single-class `onion` detector & segmenter (Target: mAP@0.5 ≥ 85% on D1-B benchmark)
- **Model 2**: 4-class defect CNN on bulb crops (`healthy`, `sprouted`, `rotten`, `mechanical_damage`) (Target: F1 ≥ 90%)
- **Export**: Lightweight ONNX models for handheld mobile inference
- **Local Download**: Programmatic transfer cell allowing instant download to your local PC without using Colab's web UI.

### Step 1: Environment & GPU Verification

In [4]:
!nvidia-smi
!pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0" "opencv-python-headless>=4.10.0" "pydantic>=2.8.0" "imagehash>=4.3.1" pandas matplotlib tqdm pyyaml

Sun Sep 20 14:05:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Download Public Datasets via Roboflow API
Enter your free Roboflow API key below to download D1-A, D1-B, and D2-A directly onto Colab's high-speed cloud disk.

In [8]:
!ls -a 

.   .config  drive    onion_model_artifacts.zip  sample_data
..  data     exports  runs


In [5]:
import os
from roboflow import Roboflow

# Enter your Roboflow Private API Key here:
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "Iro3JVh4TIt3iqgCNxpl")

os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/runs", exist_ok=True)
os.makedirs("/content/exports", exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Check D1-A and D1-B (already downloaded in earlier steps)
if not os.path.exists("/content/data/raw/D1A_instance_seg"):
    print("1. Downloading D1-A (YOLOv8 instance segmentation)... ")
    d1a_project = rf.workspace("yolo-custom-object-detection").project("instance-segmentation-wagk9")
    d1a_data = d1a_project.version(12).download("yolov8", location="/content/data/raw/D1A_instance_seg")
else:
    print("1. D1-A already present, skipping re-download.")

if not os.path.exists("/content/data/raw/D1B_onionthesis1"):
    print("\n2. Downloading D1-B (onionthesis1 - cross-source test set)... ")
    d1b_project = rf.workspace("paul-angelo-lavarias-ii").project("onionthesis1")
    d1b_data = d1b_project.version(1).download("yolov8", location="/content/data/raw/D1B_onionthesis1")
else:
    print("2. D1-B already present, skipping re-download.")

print("\n3. Downloading D2 Defect Datasets for Model 2...")
print(" - Downloading veg1-hcqsf-2 (rotten, sprout, healthy)... ")
d2_veg_project = rf.workspace("onion-grading-nx").project("veg1-hcqsf-2")
d2_veg_data = d2_veg_project.version(1).download("yolov8", location="/content/data/raw/D2_veg1")

print(" - Downloading project_onion (damaged, rotten, healthy)... ")
d2_dmg_project = rf.workspace("rishabh-raj-2k8ng").project("project_onion")
d2_dmg_data = d2_dmg_project.version(1).download("yolov8", location="/content/data/raw/D2_project_onion")

print("\n All datasets downloaded successfully into /content/data/raw/")


1. Downloading D1-A (YOLOv8 instance segmentation)... 
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/raw/D1A_instance_seg in yolov8:: 100%|██████████| 6957/6957 [00:05<00:00, 1257.28it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

2. Downloading D1-B (onionthesis1 - cross-source test set)... 
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/raw/D1B_onionthesis1 in yolov8:: 100%|██████████| 15053/15053 [00:03<00:00, 4774.85it/s]


3. Downloading D2 Defect Datasets for Model 2...
 - Downloading veg1-hcqsf-2 (rotten, sprout, healthy)... 
loading Roboflow workspace...


...


Extracting Dataset Version Zip to /content/data/raw/D2_veg1 in yolov8:: 100%|██████████| 3298/3298 [00:00<00:00, 7530.60it/s]

 - Downloading project_onion (damaged, rotten, healthy)... 
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /content/data/raw/D2_project_onion in yolov8:: 100%|██████████| 1444/1444 [00:00<00:00, 9273.15it/s]


 All datasets downloaded successfully into /content/data/raw/


### Step 3: Prepare Model 1 Dataset (Single-Class 'onion')
Unifies all masks to class `0: onion`, cleans negative labels from D1-B, and generates `data.yaml`.

In [9]:
import shutil
from pathlib import Path
import yaml

def get_onion_class_indices(raw_dir: str) -> list[int]:
    yaml_path = Path(raw_dir) / "data.yaml"
    if not yaml_path.exists():
        return [0]
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", [])
    print(f"Classes found in {Path(raw_dir).name}: {names}")
    if isinstance(names, dict):
        matches = [int(idx) for idx, name in names.items() if "not" not in str(name).lower() and "onion" in str(name).lower()]
        return matches if matches else [0]
    elif isinstance(names, list):
        matches = [idx for idx, name in enumerate(names) if "not" not in str(name).lower() and "onion" in str(name).lower()]
        return matches if matches else [0]
    return [0]

def prepare_model1_split(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, target_classes=[0]):
    Path(dst_img_dir).mkdir(parents=True, exist_ok=True)
    Path(dst_lbl_dir).mkdir(parents=True, exist_ok=True)
    
    count = 0
    for img_p in Path(src_img_dir).glob("*.jpg"):
        lbl_p = Path(src_lbl_dir) / f"{img_p.stem}.txt"
        dest_img = Path(dst_img_dir) / img_p.name
        dest_lbl = Path(dst_lbl_dir) / f"{img_p.stem}.txt"
        
        shutil.copy2(img_p, dest_img)
        if not lbl_p.exists():
            dest_lbl.touch()
            continue
            
        with open(lbl_p, "r") as f:
            lines = f.readlines()
        valid = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 6 and int(parts[0]) in target_classes:
                valid.append("0 " + " ".join(parts[1:]) + "\n")
        with open(dest_lbl, "w") as f:
            f.writelines(valid)
        count += 1
    return count

d1a_classes = get_onion_class_indices("/content/data/raw/D1A_instance_seg")
d1b_classes = get_onion_class_indices("/content/data/raw/D1B_onionthesis1")
print(f"\nTarget onion class IDs -> D1-A: {d1a_classes}, D1-B: {d1b_classes}")

# Check D1-B available split folders
d1b_root = Path("/content/data/raw/D1B_onionthesis1")
d1b_test_img = d1b_root / "test" / "images"
d1b_test_lbl = d1b_root / "test" / "labels"
if not d1b_test_img.exists():
    d1b_test_img = d1b_root / "valid" / "images"
    d1b_test_lbl = d1b_root / "valid" / "labels"
if not d1b_test_img.exists():
    d1b_test_img = d1b_root / "train" / "images"
    d1b_test_lbl = d1b_root / "train" / "labels"

print("Formatting Model 1 dataset with dynamic class resolution...")
prepare_model1_split("/content/data/raw/D1A_instance_seg/train/images", "/content/data/raw/D1A_instance_seg/train/labels", "/content/data/model1/train/images", "/content/data/model1/train/labels", target_classes=d1a_classes)
prepare_model1_split("/content/data/raw/D1A_instance_seg/valid/images", "/content/data/raw/D1A_instance_seg/valid/labels", "/content/data/model1/val/images", "/content/data/model1/val/labels", target_classes=d1a_classes)
prepare_model1_split(d1b_test_img, d1b_test_lbl, "/content/data/model1/test/images", "/content/data/model1/test/labels", target_classes=d1b_classes)

data_yaml = {
    "path": "/content/data/model1",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "onion"}
}
with open("/content/data/model1/data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print("\n Model 1 data.yaml ready at /content/data/model1/data.yaml")

Classes found in D1A_instance_seg: ['Onion']
Classes found in D1B_onionthesis1: ['Not Onion', 'Onion']

Target onion class IDs -> D1-A: [0], D1-B: [1]
Formatting Model 1 dataset with dynamic class resolution...

 Model 1 data.yaml ready at /content/data/model1/data.yaml


### Step 4: Train Model 1 (YOLOv8n-seg)
Target: **mAP@0.5 ≥ 85%** on cross-source benchmark.

In [10]:
from ultralytics import YOLO

# Load pretrained YOLOv8-nano segmentation weights
model1 = YOLO("yolov8n-seg.pt")

# Fast Fine-Tuning with RAM Caching & Early Stopping
results_m1 = model1.train(
    data="/content/data/model1/data.yaml",
    epochs=30,        # Max epochs (stops earlier if converged)
    patience=5,       # Early stopping: stop if validation mAP doesn't improve for 5 epochs
    imgsz=640,        # Set to 512 for even faster ~12s epochs
    batch=32,         # 2x batch size: doubles GPU throughput on T4
    cache="ram",      # Load images into RAM: 3x-5x faster (no repeated disk reads)
    workers=2,        # Matches Colab 2 vCPU limit (eliminates CPU bottlenecks)
    close_mosaic=5,   # Disables mosaic for final 5 epochs for fast clean convergence
    degrees=10.0,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.5,
    project="/content/runs/model1",
    name="yolov8_seg_onion",
    save=True,
    plots=True
)

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/model1/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8_seg_onion, nbs=64, nms=None, opset=No

### Step 5: Evaluate Model 1 on D1-B Cross-Source Benchmark
Tests the model on an independent author's dataset (unseen background and lighting).

In [11]:
# 1. Benchmark on D1-B test split
print("Evaluating on D1-B cross-source test set...")
metrics_m1 = model1.val(data="/content/data/model1/data.yaml", split="test")
print("=" * 60)
print(f"Model 1 D1-B Benchmark Box mAP@0.5:  {metrics_m1.box.map50:.4f}")
print(f"Model 1 D1-B Benchmark Mask mAP@0.5: {metrics_m1.seg.map50:.4f}")
print("=" * 60)

# 2. Compare against D1-A held-out test split (if present)
if Path("/content/data/raw/D1A_instance_seg/data.yaml").exists():
    try:
        metrics_d1a = model1.val(data="/content/data/raw/D1A_instance_seg/data.yaml", split="test")
        print(f"Model 1 on D1-A Test Split Mask mAP@0.5: {metrics_d1a.seg.map50:.4f}")
        print("=" * 60)
    except Exception as e:
        print(f"D1-A test split check note: {e}")

# 3. Visual Inspection: Predict and overlay masks on 3 sample D1-B test images
test_images = list(Path("/content/data/model1/test/images").glob("*.jpg"))[:3]
if test_images:
    preds = model1.predict(source=test_images, save=True, project="/content/runs/model1", name="d1b_sample_preds", conf=0.25)
    print(f"Visual sample overlays saved to: /content/runs/model1/d1b_sample_preds/")

Evaluating on D1-B cross-source test set...
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 25.8±21.1 MB/s, size: 66.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/data/model1/test/labels... 332 images, 33 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 332/332 444.9it/s 0.7s0.1s
val: /content/data/model1/test/images/IMG_2023-04-22-16-30-10-121_jpg.rf.32e7c4d4cb62ede286f4f47138ac780d.jpg: 1 duplicate labels removed
val: New cache created: /content/data/model1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.4it/s 8.8s0.2s
                   all        332      

### Step 6: Prepare Model 2 Defect Crops (4 Core Classes)
Extracts individual bulb crops from D2-A and categorizes them into `healthy`, `sprouted`, `rotten`, `mechanical_damage`.

In [12]:
import cv2
from pathlib import Path
import random
import yaml

def find_label_file(img_path: Path) -> Path | None:
    # 1. Same directory
    lbl = img_path.with_suffix(".txt")
    if lbl.exists(): return lbl
    # 2. Path replacing /images/ with /labels/
    parts = list(img_path.parts)
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        lbl = Path(*parts).with_suffix(".txt")
        if lbl.exists(): return lbl
    # 3. Sibling labels folder
    lbl = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    if lbl.exists(): return lbl
    return None

def map_defect_class(raw_name: str) -> str | None:
    name = str(raw_name).strip().lower()
    if "rot" in name: return "rotten"
    if "sprout" in name or "root" in name: return "sprouted"
    if "damag" in name or "cut" in name: return "mechanical_damage"
    if "health" in name or name == "onion": return "healthy"
    return None

classes = ["healthy", "sprouted", "rotten", "mechanical_damage"]
for s in ["train", "val", "test"]:
    for c in classes:
        Path(f"/content/data/model2/{s}/{c}").mkdir(parents=True, exist_ok=True)

d2_source_dirs = [
    Path("/content/data/raw/D2_veg1"),
    Path("/content/data/raw/D2_project_onion"),
]

crops_by_class = {c: [] for c in classes}
img_extensions = {".jpg", ".jpeg", ".png"}

for src_dir in d2_source_dirs:
    if not src_dir.exists(): continue
    
    # Read data.yaml for this dataset
    yaml_p = src_dir / "data.yaml"
    raw_names = []
    if yaml_p.exists():
        with open(yaml_p) as f:
            d_cfg = yaml.safe_load(f)
        raw_names = d_cfg.get("names", [])
        if isinstance(raw_names, dict):
            raw_names = [raw_names[k] for k in sorted(raw_names.keys())]
    
    print(f"\nProcessing {src_dir.name} with classes: {raw_names}")
    images = [p for p in src_dir.rglob("*") if p.suffix.lower() in img_extensions]
    print(f"Found {len(images)} images in {src_dir.name}. Extracting crops...")
    
    for img_p in images:
        lbl_p = find_label_file(img_p)
        if not lbl_p or not lbl_p.exists(): continue
        img = cv2.imread(str(img_p))
        if img is None: continue
        h, w = img.shape[:2]
        
        with open(lbl_p) as f: lines = f.readlines()
        for idx, l in enumerate(lines):
            p = l.strip().split()
            if len(p) < 5: continue
            cid = int(p[0])
            cname = raw_names[cid] if raw_names and cid < len(raw_names) else str(cid)
            target_cls = map_defect_class(cname)
            if not target_cls: continue
            
            cx, cy, bw, bh = float(p[1]), float(p[2]), float(p[3]), float(p[4])
            # 8% margin padding
            x1, y1 = max(0, int((cx - bw*0.54)*w)), max(0, int((cy - bh*0.54)*h))
            x2, y2 = min(w, int((cx + bw*0.54)*w)), min(h, int((cy + bh*0.54)*h))
            if x2 > x1 and y2 > y1:
                crop = cv2.resize(img[y1:y2, x1:x2], (224, 224), interpolation=cv2.INTER_AREA)
                crops_by_class[target_cls].append((crop, f"{src_dir.name}_{img_p.stem}_c{idx}.jpg"))

# Balance and save train (70%), val (15%), test (15%)
print("\nBalancing and saving crops across splits:")
for c, items in crops_by_class.items():
    random.shuffle(items)
    # Cap healthy crops to at most 500 for balance
    if c == "healthy" and len(items) > 500:
        items = items[:500]
    n1 = int(len(items)*0.70)
    n2 = int(len(items)*0.85)
    for crop, name in items[:n1]: cv2.imwrite(f"/content/data/model2/train/{c}/{name}", crop)
    for crop, name in items[n1:n2]: cv2.imwrite(f"/content/data/model2/val/{c}/{name}", crop)
    for crop, name in items[n2:]: cv2.imwrite(f"/content/data/model2/test/{c}/{name}", crop)
    print(f"  - {c}: {len(items)} crops (train: {n1}, val: {n2-n1}, test: {len(items)-n2})")

print("\n Model 2 ImageFolder dataset ready at /content/data/model2")



Processing D2_veg1 with classes: ['double_split', 'onion', 'rotten', 'sprout']
Found 1643 images in D2_veg1. Extracting crops...

Processing D2_project_onion with classes: ['Damaged', 'Healthy', 'Rotten']
Found 720 images in D2_project_onion. Extracting crops...

Balancing and saving crops across splits:
  - healthy: 500 crops (train: 350, val: 75, test: 75)
  - sprouted: 536 crops (train: 375, val: 80, test: 81)
  - rotten: 1396 crops (train: 977, val: 209, test: 210)
  - mechanical_damage: 237 crops (train: 165, val: 36, test: 36)

 Model 2 ImageFolder dataset ready at /content/data/model2


### Step 7: Train Model 2 (Defect Classifier)
Target: **F1 ≥ 90%** on defect classes.

In [13]:
model2 = YOLO("yolov8n-cls.pt")

# Fast Classifier Training with RAM Caching & Early Stopping
results_m2 = model2.train(
    data="/content/data/model2",
    epochs=25,        # Max epochs
    patience=5,       # Early stopping: stop if validation metric plateaus
    imgsz=224,        # Standard crop input size
    batch=64,         # Fast batch size on T4 for 224x224 crops
    cache="ram",      # Load all crops into RAM
    workers=2,        # Matches Colab 2 vCPUs
    project="/content/runs/model2",
    name="defect_classifier_4class",
    save=True,
    plots=True
)

metrics_m2 = model2.val(data="/content/data/model2", split="test")
print(f"\nModel 2 Top-1 Test Accuracy: {metrics_m2.top1:.4f}")

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/model2, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=defect_classifier_4class, nbs=64, nms=None, opset=None

### Step 8: Export Both Models to ONNX for Mobile Deployment

In [14]:
from ultralytics import YOLO
import glob

# Ensure model1 is loaded from best weights if not already in session
if 'model1' not in locals() or model1 is None:
    m1_pts = sorted(glob.glob('/content/runs/model1/**/weights/best.pt', recursive=True))
    if m1_pts:
        model1 = YOLO(m1_pts[-1])
        print(f"Loaded Model 1 from: {m1_pts[-1]}")

# Ensure model2 is loaded from best weights if not already in session
if 'model2' not in locals() or model2 is None:
    m2_pts = sorted(glob.glob('/content/runs/model2/**/weights/best.pt', recursive=True))
    if m2_pts:
        model2 = YOLO(m2_pts[-1])
        print(f"Loaded Model 2 from: {m2_pts[-1]}")

# Export Model 1 (detection + segmentation) with dynamic shape for mobile resolution flexibility
onnx_m1 = model1.export(format="onnx", dynamic=True)
# Export Model 2 (defect classifier) at fixed 224x224
onnx_m2 = model2.export(format="onnx", imgsz=224)

print(f"\n Exported Model 1 ONNX: {onnx_m1}")
print(f" Exported Model 2 ONNX: {onnx_m2}")


Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)


💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs

PyTorch: starting from '/content/runs/model1/yolov8_seg_onion/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 37, 8400), (1, 32, 160, 160)) (6.4 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 351ms
Prepared 5 packages in 572ms
Uninstalled 1 package in 21ms
Installed 5 packages in 37ms
 + colorama==0.4.6
 + onnx==1.23.0
 + onnxruntime==1.30.0
 + onnxslim==0.1.96
 - protobuf==5.29.6
 + protobuf==7.36.2

requirements: AutoUpdate success ✅ 1.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.0 opset 18..

### Step 9: Download Trained Models Directly to Your PC (No Colab Web UI Needed)
Run the cell below. It packages all `.pt` weights, `.onnx` models, and evaluation plots, uploads them, and gives you an exact 1-line `curl` command to run in your local PC terminal.

In [16]:
import os
import glob
import shutil
import subprocess

# 1. Organize exports directory
os.makedirs("/content/exports/weights", exist_ok=True)
os.makedirs("/content/exports/onnx", exist_ok=True)
os.makedirs("/content/exports/eval_plots", exist_ok=True)
os.makedirs("/content/exports/training_runs", exist_ok=True)

# Copy latest best.pt for Model 1 and Model 2
m1_pts = sorted(glob.glob("/content/runs/model1/**/weights/best.pt", recursive=True), key=os.path.getmtime)
m2_pts = sorted(glob.glob("/content/runs/model2/**/weights/best.pt", recursive=True), key=os.path.getmtime)

if m1_pts:
    shutil.copy(m1_pts[-1], "/content/exports/weights/model1_yolov8_seg.pt")
    print(f" Copied Model 1 weights from: {m1_pts[-1]}")
if m2_pts:
    shutil.copy(m2_pts[-1], "/content/exports/weights/model2_defect_cls.pt")
    print(f" Copied Model 2 weights from: {m2_pts[-1]}")

# Copy exported ONNX files
m1_onnx = sorted(glob.glob("/content/runs/model1/**/weights/*.onnx", recursive=True) + glob.glob("/content/*.onnx"), key=os.path.getmtime)
m2_onnx = sorted(glob.glob("/content/runs/model2/**/weights/*.onnx", recursive=True), key=os.path.getmtime)

for f in m1_onnx:
    if "seg" in f.lower() or "best" in f.lower():
        shutil.copy(f, "/content/exports/onnx/model1_yolov8_seg.onnx")
        print(f" Copied Model 1 ONNX: {f}")
        break
for f in m2_onnx:
    shutil.copy(f, "/content/exports/onnx/model2_defect_cls.onnx")
    print(f" Copied Model 2 ONNX: {f}")
    break

# Copy all training evaluation plots, loss curves, confusion matrices, and metrics CSVs
plots_copied = 0
for ext in ["*.png", "*.jpg", "*.csv", "*.yaml"]:
    for p in glob.glob(f"/content/runs/**/{ext}", recursive=True):
        dest_name = os.path.basename(p)
        parent_tag = os.path.basename(os.path.dirname(p))
        shutil.copy(p, f"/content/exports/eval_plots/{parent_tag}_{dest_name}")
        plots_copied += 1
print(f" Bundled {plots_copied} training plots, loss curves, confusion matrices, and metrics CSVs into eval_plots.")

# Copy the entire runs directory structure so all raw subfolders and logs are preserved
!cp -r /content/runs /content/exports/training_runs/

# 2. Zip archive containing /content/exports (weights, onnx, eval_plots, and training_runs)
archive_path = "/content/onion_model_artifacts.zip"
!zip -r {archive_path} /content/exports

# 3. Programmatic CLI Download via transfer.sh
print("\nUploading complete package (weights + ONNX + all plots & graphs) to transfer.sh...")
res = subprocess.run(f"curl --upload-file {archive_path} https://transfer.sh/onion_model_artifacts.zip", shell=True, capture_output=True, text=True)
url = res.stdout.strip()

print("\n" + "="*80)
print(" TRAINING COMPLETE! DOWNLOAD ALL ARTIFACTS TO YOUR PC WITH THIS COMMAND:")
print("="*80)
print(f"\nmkdir -p ~/Projects/Onion_grading_new/runs && curl -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip {url} && unzip -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip -d ~/Projects/Onion_grading_new/runs/\n")
print("="*80)


 Copied Model 1 weights from: /content/runs/model1/yolov8_seg_onion/weights/best.pt
 Copied Model 2 weights from: /content/runs/model2/defect_classifier_4class/weights/best.pt
 Copied Model 1 ONNX: /content/runs/model1/yolov8_seg_onion/weights/best.onnx
 Copied Model 2 ONNX: /content/runs/model2/defect_classifier_4class/weights/best.onnx
 Bundled 83 training plots, loss curves, confusion matrices, and metrics CSVs into eval_plots.
updating: content/exports/ (stored 0%)
updating: content/exports/training_runs/ (stored 0%)
updating: content/exports/onnx/ (stored 0%)
updating: content/exports/weights/ (stored 0%)
updating: content/exports/eval_plots/ (stored 0%)
  adding: content/exports/training_runs/runs/ (stored 0%)
  adding: content/exports/training_runs/runs/model2/ (stored 0%)
  adding: content/exports/training_runs/runs/model2/defect_classifier_4class/ (stored 0%)
  adding: content/exports/training_runs/runs/model2/defect_classifier_4class/train_batch0.jpg (deflated 3%)
  adding: c

### Step 10: Copy Artifacts to Google Drive (Or Download Directly)
If Drive is mounted, this saves the zip to `MyDrive/onion_grading_runs/onion_model_artifacts.zip`.


In [17]:
import os
import shutil

drive_dest = "/content/drive/MyDrive/onion_grading_runs"
zip_src = "/content/onion_model_artifacts.zip"

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(zip_src, f"{drive_dest}/onion_model_artifacts.zip")
    print("="*80)
    print(f" SUCCESS! Artifacts copied to Google Drive at:")
    print(f" {drive_dest}/onion_model_artifacts.zip")
    print("="*80)
else:
    print("="*80)
    print(" Google Drive is not mounted yet.")
    print(" To mount Drive in 5 seconds:")
    print(" 1. Open your browser tab where Colab is running.")
    print(" 2. In the left panel, click the Folder icon -> click the 'Mount Drive' button (folder with Drive logo).")
    print(" 3. Click 'Connect to Google Drive' on the popup.")
    print(" 4. Re-run this cell to immediately copy the file into your Drive!")
    print("="*80)


 SUCCESS! Artifacts copied to Google Drive at:
 /content/drive/MyDrive/onion_grading_runs/onion_model_artifacts.zip


### (Alternative) Direct Download via Cloudflare Tunnel (No Browser / No Drive needed)
Generates a direct, secure HTTPS link to download `onion_model_artifacts.zip` straight from Colab into your PC terminal.


In [24]:
import os
import subprocess
import threading
import http.server
import socketserver

# 1. Install static cloudflared binary
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# 2. Start quiet local HTTP server on port 8080
!fuser -k 8080/tcp 2>/dev/null || true
os.chdir("/content")
class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass

server = socketserver.TCPServer(("", 8080), QuietHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

# 3. Start Cloudflare tunnel
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8080"], stderr=subprocess.PIPE, text=True)
tunnel_url = None
for line in p.stderr:
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part and part.startswith("http"):
                tunnel_url = part.strip()
                break
        if tunnel_url: break

print("\n" + "="*80)
print(" DIRECT DOWNLOAD LINK ACTIVE:")
print(f" {tunnel_url}/onion_model_artifacts.zip")
print("="*80)
print(f"\nRun this command in your PC terminal to download:\n")
print(f"mkdir -p ~/Projects/Onion_grading_new/runs && curl -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip {tunnel_url}/onion_model_artifacts.zip && unzip -o ~/Projects/Onion_grading_new/runs/onion_model_artifacts.zip -d ~/Projects/Onion_grading_new/runs/\n")


: 

In [19]:
from google.colab import drive

drive.unmount("/content/drive")


AttributeError: module 'google.colab.drive' has no attribute 'unmount'